
# ML & Feature store: Leverage Databricks Feature Tables to share cohorts

### Register Feature Store Tables


<img src="https://github.com/QuentinAmbard/databricks-demo/raw/main/product_demos/mlops-end2end-flow-feature-store.png" style="float:right" width="650" />

Databricks provide Feature Store capabilities, simplifying operational work and increasing discoverability among your Data Scientist team, accelerating Analysis.

Under the hood, feature store are backed by a Delta Lake table. This will allow discoverability and reusability of our feature across our organization, increasing team efficiency. <br/>
These tables can be backed with Online tables (such as DynamoDB or CosmoDB), provinding instant feature lookup and transformations (ms) for realtime inferences.

Databricks Feature Store brings advanced capabilities to accelerate and simplify your ML journey, such as point in time support and online-store, fetching your features within ms for real time Serving. 

### Why use Databricks Feature Store?

Databricks Feature Store is fully integrated with other components of Databricks.

* **Discoverability**. The Feature Store UI, accessible from the Databricks workspace, lets you browse and search for existing features.

* **Lineage**. When you create a feature table with Feature Store, the data sources used to create the feature table are saved and accessible. For each feature in a feature table, you can also access the models, notebooks, jobs, and endpoints that use the feature.

* **Batch and Online feature lookup for real time serving**. When you use features from Feature Store to train a model, the model is packaged with feature metadata. When you use the model for batch scoring or online inference, it automatically retrieves features from Feature Store. The caller does not need to know about them or include logic to look up or join features to score new data. This makes model deployment and updates much easier.

* **Point-in-time lookups**. Feature Store supports time series and event-based use cases that require point-in-time correctness.


For more details about Databricks Feature Store, run `dbdemos.install('feature-store')`

<!-- Collect usage data (view). Remove it to disable collection. View README for more details.  -->
<img width="1px" src="https://ppxrzfxige.execute-api.us-west-2.amazonaws.com/v1/analytics?category=lakehouse&org_id=2162748966026566&notebook=%2F04-Data-Science-ML%2F04.6-EXTRA-Feature-Store-ML-patient-readmission&demo_name=lakehouse-hls-readmission&event=VIEW&path=%2F_dbdemos%2Flakehouse%2Flakehouse-hls-readmission%2F04-Data-Science-ML%2F04.6-EXTRA-Feature-Store-ML-patient-readmission&version=1">

In [0]:
%pip install databricks-sdk==0.39.0 mlflow==2.19.0 databricks-feature-store==0.17.0
dbutils.library.restartPython()

  Using cached databricks_sdk-0.39.0-py3-none-any.whl.metadata (38 kB)
  Using cached mlflow-2.19.0-py3-none-any.whl.metadata (30 kB)
  Using cached mlflow_skinny-2.19.0-py3-none-any.whl.metadata (31 kB)
  Using cached docker-7.1.0-py3-none-any.whl.metadata (3.8 kB)
  Using cached graphene-3.4.3-py2.py3-none-any.whl.metadata (6.9 kB)
  Using cached graphql_relay-3.2.0-py3-none-any.whl.metadata (12 kB)
Using cached databricks_sdk-0.39.0-py3-none-any.whl (622 kB)
Using cached mlflow-2.19.0-py3-none-any.whl (27.4 MB)
Using cached mlflow_skinny-2.19.0-py3-none-any.whl (5.9 MB)
Using cached docker-7.1.0-py3-none-any.whl (147 kB)
Using cached graphene-3.4.3-py2.py3-none-any.whl (114 kB)
Using cached graphql_relay-3.2.0-py3-none-any.whl (16 kB)
  Attempting uninstall: databricks-sdk
    Found existing installation: databricks-sdk 0.30.0
    Not uninstalling databricks-sdk at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-6398a2

In [0]:
%run ../_resources/00-setup $reset_all_data=false

USE CATALOG `main__build`
using catalog.database `main__build`.`dbdemos_hls_readmission`


data already existing. Run with reset_all_data=true to force a data cleanup for your local demo.


## How to use this notebook
This notebook is added as extra, and would typically be used as replacement instead of the first [04.1-Feature-Engineering-patient-readmission]($./04.1-Feature-Engineering-patient-readmission) notebook.

For more details on feature store, install `dbdemos.install('feature-store')`

## Patient Features
Let's start with our patient features. We'll encode our categories leveraging pandas on spark API and the `get_dummies()` function.

Pandas On Spark (previously Koalas) makes transformation at scale easy, leveraging the well-known pandas API with spark distributed backend.

In [0]:
import pyspark.pandas as ps

# Define Patient Features logic
def compute_pat_features(data):
  data = data.pandas_api()
  data = ps.get_dummies(data, columns=['MARITAL', 'RACE', 'ETHNICITY', 'GENDER'],dtype = 'int64').to_spark()
  return data

In [0]:
pat_features_df = compute_pat_features(spark.table('patients').dropDuplicates(["Id"]))
pat_features_df.display()

Id,BIRTHDATE,DEATHDATE,SSN,DRIVERS,PASSPORT,PREFIX,FIRST,LAST,SUFFIX,MAIDEN,BIRTHPLACE,ADDRESS,CITY,STATE,COUNTY,FIPS,ZIP,LAT,LON,HEALTHCARE_EXPENSES,HEALTHCARE_COVERAGE,INCOME,_rescued_data,MARITAL_D,MARITAL_M,MARITAL_S,MARITAL_W,RACE_asian,RACE_black,RACE_hawaiian,RACE_native,RACE_other,RACE_white,ETHNICITY_hispanic,ETHNICITY_nonhispanic,GENDER_F,GENDER_M
0044c2e0-1017-01d4-223f-2225e39f5179,1970-08-26,null,999-25-1783,S99954416,X70937992X,Mrs.,Aurore252,Herman763,null,Schneider199,Lowell Massachusetts US,1023 Cole Spur Suite 59,Sunderland,Massachusetts,Franklin County,null,0,42.47310542066492,-72.52425553116522,747944.13,417783.96,36832,null,0,1,0,0,0,0,0,0,0,1,0,1,1,0
00474dfe-c083-4000-0a76-18dca8467373,1993-07-20,null,999-45-2437,S99910958,X77042108X,Mr.,Johnnie679,Hirthe744,null,null,Worcester Massachusetts US,122 Hauck Ramp Unit 29,Sutton,Massachusetts,Worcester County,null,0,42.13367945968884,-71.76318769436119,50277.54,85.33,140907,null,0,1,0,0,0,0,0,0,0,1,0,1,0,1
004af2f6-7fcf-6c2e-e8ba-d34b36851b0d,1922-08-22,1989-08-30,999-51-9298,S99942796,X39711714X,Mr.,Florencio463,VonRueden376,null,null,Framingham Massachusetts US,787 Spencer Corner Suite 48,Dracut,Massachusetts,Middlesex County,null,0,42.658088527988895,-71.31563744419455,126493.0,13740.54,74796,null,0,1,0,0,0,0,0,0,0,1,0,1,0,1
004f14e7-dc30-71bb-c934-50b5c632ed9b,1996-05-25,null,999-50-6379,S99915208,X64148905X,Ms.,Polly738,Stark857,null,null,Boston Massachusetts US,501 Corwin Quay,Leominster,Massachusetts,Worcester County,25027.0,1453,42.53774202399741,-71.78140246412941,264780.03,1976.46,80354,null,0,0,0,0,0,0,0,0,0,1,0,1,1,0
00c1d7e1-467b-286b-272f-077cf779db80,2014-06-28,null,999-37-5988,null,null,null,Blake449,Wiza601,null,null,Lawrence Massachusetts US,1035 Jacobs Walk Unit 38,Lexington,Massachusetts,Middlesex County,25017.0,2420,42.46509782838845,-71.27135616483878,16800.44,5584.72,194209,null,0,0,0,0,0,0,0,0,0,1,0,1,1,0
00c83721-8756-f0a9-4225-34e6879bbf22,1937-01-07,null,999-72-4800,S99918908,X77727216X,Mr.,Milton509,Cummerata161,null,null,Wilbraham Massachusetts US,161 Welch Forge Unit 43,Salisbury,Massachusetts,Essex County,25009.0,1952,42.86974344988821,-70.79384492275234,122354.78,222438.31,134997,null,0,1,0,0,0,0,0,0,0,1,1,0,0,1
00dfbb9f-a897-1252-e69d-0c3704621753,2011-05-29,null,999-43-5828,null,null,null,Logan497,Block661,null,null,Lowell Massachusetts US,411 Wunsch Extension,Somerville,Massachusetts,Middlesex County,25017.0,2138,42.39176189198144,-71.15565174335529,17258.93,0.0,148745,null,0,0,0,0,1,0,0,0,0,0,0,1,0,1
00dfe9de-0c81-ef2a-a9f9-8e789d3a198b,1976-08-18,null,999-68-6506,S99967532,X26888394X,Mr.,Lyle846,Braun514,null,null,Boston Massachusetts US,128 Kulas Port Unit 21,Taunton,Massachusetts,Bristol County,25005.0,2767,41.84916922251937,-71.0068692362087,69541.81,27.65,50100,null,0,0,0,1,0,0,0,0,0,1,0,1,0,1
00f68dcf-9eaf-e774-3a1b-38e669be1452,1963-11-18,2020-11-14,999-24-9023,S99933305,X36739090X,Mr.,Bruno518,Wolff180,null,null,Revere Massachusetts US,751 Klein Track Apt 63,Arlington,Massachusetts,Middlesex County,25017.0,2474,42.40459992198894,-71.22040223311642,871742.4,3313137.08,29532,null,0,1,0,0,1,0,0,0,0,0,0,1,0,1
00f88720-308b-272f-f3b3-1121497acde0,1983-12-09,null,999-97-8789,S99948214,X1912185X,Ms.,Necole468,Upton904,MD,null,Worcester Massachusetts US,864 Miller Road Apt 54,Blackstone,Massachusetts,Worcester County,null,0,42.03645890234569,-71.54658230977826,243194.64,576040.87,126114,null,0,0,1,0,0,0,0,0,0,1,0,1,1,0


## Creating the Feature Table
Now that our features are ready, let's save them as a Feature Table

In [0]:
# Instantiate the Feature Store Client
from databricks import feature_store

# Instantiate the Feature Store Client
fs = feature_store.FeatureStoreClient()

drop_fs_table(f'{dbName}.pat_features')

pat_feature_table = fs.create_table(
  name=f'{dbName}.pat_features',
  primary_keys=['Id'],
  df=pat_features_df,
  description='Base Features from the Patient Table'
)

fs.write_table(df=pat_features_df, name=f'{dbName}.pat_features', mode='overwrite')

2025/11/03 22:44:11 WARNING databricks.ml_features._compute_client._compute_client: Deleting a feature table can lead to unexpected failures in upstream producers and downstream consumers (models, endpoints, and scheduled jobs).
2025/11/03 22:44:14 INFO databricks.ml_features._compute_client._compute_client: Setting columns ['Id'] of table 'main.dbdemos_hls_readmission.pat_features' to NOT NULL.
2025/11/03 22:44:16 INFO databricks.ml_features._compute_client._compute_client: Setting Primary Keys constraint ['Id'] on table 'main.dbdemos_hls_readmission.pat_features'.
2025/11/03 22:44:20 INFO databricks.ml_features._compute_client._compute_client: Created feature table 'main.dbdemos_hls_readmission.pat_features'.


## Encounter Feature Table
We can now repeat the same steps with a new encounter Feature Table

In [0]:
import pyspark.pandas as ps
def compute_enc_features(data):
  data = data.dropDuplicates(["Id"])
  data = data.withColumn('enc_length', F.unix_timestamp(col('stop'))- F.unix_timestamp(col('start')))
  data = data.pandas_api()
#   return data
  data = ps.get_dummies(data, columns=['ENCOUNTERCLASS'],dtype = 'int64').to_spark()
  
  return (
    data
    .select(
      col('Id').alias('ENCOUNTER_ID'),
      'BASE_ENCOUNTER_COST',
      'TOTAL_CLAIM_COST',
      'PAYER_COVERAGE',
      'enc_length',
      'ENCOUNTERCLASS_ambulatory',
      'ENCOUNTERCLASS_emergency',
      'ENCOUNTERCLASS_hospice',
      'ENCOUNTERCLASS_inpatient',
      'ENCOUNTERCLASS_outpatient',
      'ENCOUNTERCLASS_wellness',
    )
  )

In [0]:
enc_features_df = compute_enc_features(spark.table('encounters'))

drop_fs_table(f'{dbName}.enc_features')

#Note: You might need to delete the FS table using the UI
enc_feature_table = fs.create_table(
  name=f'{dbName}.enc_features',
  primary_keys=['ENCOUNTER_ID'],
  df=enc_features_df,
  description='Base and derived features from the Encounter Table'
)

fs.write_table(df=enc_features_df, name=f'{dbName}.enc_features', mode='overwrite')

2025/11/03 22:44:32 WARNING databricks.ml_features._compute_client._compute_client: Deleting a feature table can lead to unexpected failures in upstream producers and downstream consumers (models, endpoints, and scheduled jobs).
2025/11/03 22:44:34 INFO databricks.ml_features._compute_client._compute_client: Setting columns ['ENCOUNTER_ID'] of table 'main.dbdemos_hls_readmission.enc_features' to NOT NULL.
2025/11/03 22:44:35 INFO databricks.ml_features._compute_client._compute_client: Setting Primary Keys constraint ['ENCOUNTER_ID'] on table 'main.dbdemos_hls_readmission.enc_features'.
2025/11/03 22:44:47 INFO databricks.ml_features._compute_client._compute_client: Created feature table 'main.dbdemos_hls_readmission.enc_features'.


### Adding extra Feature Table for each patient: age at encounter

In [0]:
def compute_age_at_enc(encounters, patients):
  #for the demo, we drop potential duplicates in case data is re-inserted in the ingestion part
  encounters = encounters.dropDuplicates(["Id"])
  patients = patients.dropDuplicates(["Id"])
  return (
    encounters
    .join(patients, patients['id'] == encounters['PATIENT'])
    .select(
      encounters.Id.alias('encounter_id'),
      patients.Id.alias('patient_id'),
      ((F.datediff(col('START'), col('BIRTHDATE'))) / 365.25).alias('age_at_encounter')
    )
  )

In [0]:
aae_features_df = compute_age_at_enc(spark.table('encounters'), spark.table('patients'))

drop_fs_table(f'{dbName}.age_at_enc_features')

#Note: You might need to delete the FS table using the UI
aae_feature_table = fs.create_table(
  name=f'{dbName}.age_at_enc_features',
  primary_keys=['encounter_id'],
  df=aae_features_df,
  description='determine the age of the patient at the time of the encounter'
)

fs.write_table(df=aae_features_df, name=f'{dbName}.age_at_enc_features', mode='overwrite')

2025/11/03 22:44:58 WARNING databricks.ml_features._compute_client._compute_client: Deleting a feature table can lead to unexpected failures in upstream producers and downstream consumers (models, endpoints, and scheduled jobs).
2025/11/03 22:45:00 INFO databricks.ml_features._compute_client._compute_client: Setting columns ['encounter_id'] of table 'main.dbdemos_hls_readmission.age_at_enc_features' to NOT NULL.
2025/11/03 22:45:01 INFO databricks.ml_features._compute_client._compute_client: Setting Primary Keys constraint ['encounter_id'] on table 'main.dbdemos_hls_readmission.age_at_enc_features'.
2025/11/03 22:45:09 INFO databricks.ml_features._compute_client._compute_client: Created feature table 'main.dbdemos_hls_readmission.age_at_enc_features'.



%md
### Building a dataset from the feature store

Now that our feature tables are created, we can query them to build training datasets.

This is done by building Feature Lookups, specifying the key used to retrive the data. 

This is conceptually close to a SQL JOIN on the key between between your label dataset and the feature store tables. For offline batch this is done with Spark as a backend, and for realtime feature lookup an Key/Value backend can be used to request the features for a given key.

In [0]:
#Note that labels should not be part of the features
from pyspark.sql import Window
# Let's create our label: we'll predict the  30 days readmission risk
windowSpec = Window.partitionBy("PATIENT").orderBy("START")
labels = spark.table('encounters').select("PATIENT", "Id", "START", "STOP") \
              .withColumn('30_DAY_READMISSION', F.when(col('START').cast('long') - F.lag(col('STOP')).over(windowSpec).cast('long') < 30*24*60*60, 1).otherwise(0))
display(labels)

PATIENT,Id,START,STOP,30_DAY_READMISSION
004f14e7-dc30-71bb-c934-50b5c632ed9b,9f9f98cb-c55c-2255-8adb-28e189ef1ffa,2010-06-26T19:27:33Z,2010-06-26T19:49:27Z,0
004f14e7-dc30-71bb-c934-50b5c632ed9b,8b78cd72-a535-43ed-4c39-ac21b9f546a7,2011-12-17T19:27:33Z,2011-12-17T19:52:54Z,0
004f14e7-dc30-71bb-c934-50b5c632ed9b,4295c416-05e9-63c5-5ded-7534d0f03740,2013-05-13T23:10:30Z,2013-05-13T23:25:30Z,0
004f14e7-dc30-71bb-c934-50b5c632ed9b,0975986f-90bd-8539-c9b3-7ae6c6f5a590,2013-07-13T19:27:33Z,2013-07-13T19:42:33Z,0
004f14e7-dc30-71bb-c934-50b5c632ed9b,6f1de637-751c-c3bf-10e1-612571f4f338,2014-07-19T19:27:33Z,2014-07-19T20:10:46Z,0
004f14e7-dc30-71bb-c934-50b5c632ed9b,20fbd2d7-72f5-5309-c652-d59f1e07627c,2015-05-03T23:10:30Z,2015-05-03T23:30:21Z,0
004f14e7-dc30-71bb-c934-50b5c632ed9b,6248ba13-74da-4965-cff8-d23d9f35a402,2015-07-25T19:27:33Z,2015-07-25T20:21:00Z,0
004f14e7-dc30-71bb-c934-50b5c632ed9b,9a90ea6d-49ea-59ec-8b08-c18510c8ad09,2016-05-28T19:27:33Z,2016-05-28T20:10:36Z,0
004f14e7-dc30-71bb-c934-50b5c632ed9b,4fa253d7-059a-7065-15c8-e5ce88870b2d,2016-07-30T19:27:33Z,2016-07-30T20:37:30Z,0
004f14e7-dc30-71bb-c934-50b5c632ed9b,9058fbbd-984f-4176-de72-9e4ac85a73be,2016-09-17T19:27:33Z,2016-09-17T19:42:33Z,0


In [0]:
from databricks.feature_store import FeatureLookup
patient_feature_lookups = [
   FeatureLookup( 
     table_name = f'{dbName}.pat_features',
     feature_names = [
      'MARITAL_M',
      'MARITAL_S',
      'RACE_asian',
      'RACE_black',
      'RACE_hawaiian',
      'RACE_other',
      'RACE_white',
      'ETHNICITY_hispanic',
      'ETHNICITY_nonhispanic',
      'GENDER_F',
      'GENDER_M',
      'INCOME'],
     lookup_key = ["PATIENT"]
   )
]
 
encounter_feature_lookups = [
   FeatureLookup( 
     table_name = f'{dbName}.enc_features',
     feature_names = ['BASE_ENCOUNTER_COST', 'TOTAL_CLAIM_COST', 'PAYER_COVERAGE', 'enc_length', 'ENCOUNTERCLASS_ambulatory', 'ENCOUNTERCLASS_emergency', 'ENCOUNTERCLASS_hospice', 'ENCOUNTERCLASS_inpatient', 'ENCOUNTERCLASS_outpatient', 'ENCOUNTERCLASS_wellness',],
     lookup_key = ["Id"]
   )
]

age_at_enc_feature_lookups = [
   FeatureLookup( 
     table_name = f'{dbName}.age_at_enc_features',
     feature_names = ['age_at_encounter'],
     lookup_key = ["Id"]
   )
]

### Use Feature Store to Create Dataset Based on Lookups

In [0]:
fs = feature_store.FeatureStoreClient()
training_set = fs.create_training_set(
  labels,
  feature_lookups = patient_feature_lookups + encounter_feature_lookups + age_at_enc_feature_lookups,
  label = "30_DAY_READMISSION",
  exclude_columns = ['START', 'STOP']
)

training_df = training_set.load_df()

## Congrats! our training dataset is ready, retreiving features from our Feature Tables.

We now continue our ML steps such as calling AutoML with the training Dataset.

For more details on Databricks Feature Store and more advanced capabilities (Online Store for realtime lookup, streaming updates, timeseries processing...), install the feature store demo: `dbdemos.install('feature-store')` 

In [0]:
#from databricks import automl
#summary = automl.classify(training_dataset.select(feature_names), target_col="30_DAY_READMISSION", primary_metric="roc_auc", timeout_minutes=6)
# ...
# See 04.2-AutoML-patient-admission-risk for more details on how to deploy the AutoML model.


## Next: Continue the model deployment and serving 

Our Feature Store tables are now available for all the organization and can be leverage to train models on any patient cohorts. 

Data Scientists can easily explore these tables, and leverage them to bootstrap their experience.

Once your feature store is deployed, the same steps apply for model training and inference. See the [04.3-Batch-Scoring-patient-readmission]($./04.3-Batch-Scoring-patient-readmission) for the next steps.